# Erste Schritte 

## Erstes Environement mit Random actions

In [ ]:
import gymnasium as gym
env = gym.make('CartPole-v1', render_mode='human')
obs, info = env.reset()  #obs ist der aktuelle zustand des Wagens und info die zusatzinfo
for i in range(10000):
    action = env.action_space.sample()# macht random aktionen
    obs, reward, terminated, truncated, info = env.step(action)

    if terminated or truncated:
        obs,info = env.reset()
    print(obs)

## Lernen mithilfe der Q gleichung
#### Da wir in dem ersten beispiel unendliche möglichkeiten bei den States haben müssen wir auf ein Environement mit fixen States wechseln

In [ ]:
import gymnasium as gym
import numpy as np
import pygame


alpha= 0.1 #lernrate
gamma=0.99 # disoount


epsilon=1 
epsilon_minimum= 0.01
epsilon_decay= 0.995

q_table = np.zeros([16, 4] )
env = gym.make('FrozenLake-v1', is_slippery=False)
state, info = env.reset()



running = True
for episode in range(20000):
    state, info = env.reset()
    done=False
    while not done:
        
        #aufgabe 3
        if np.random.random()< epsilon:
            action= env.action_space.sample()  
        else:
            action=np.argmax(q_table[state]) 


        new_state, reward, terminated, truncated, info = env.step(action)

        omega= reward + gamma * np.max(q_table[new_state]) - q_table[state, action] #berechnung des TD errors
        q_table[state, action]= q_table[state,action]+alpha * omega 


        state = new_state

        if terminated or truncated:
            done=True
        
    epsilon=max(epsilon_minimum, epsilon * epsilon_decay)
np.save("q_table.npy", q_table)
env.close()


### Testing

In [20]:
import gymnasium as gym
import numpy as np
import pygame

q_table = np.load('q_table.npy')
env = gym.make('FrozenLake-v1', render_mode='human', is_slippery=False)
state, info = env.reset()

running = True
while running:
    action = np.argmax(q_table[state])
    new_state, reward, terminated, truncated, info = env.step(action)
    state=new_state
    if terminated or truncated:
        obs, info = env.reset()

    for event in pygame.event.get():
        if event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE:
            running = False

env.close()






# Jetzt kommt der Cartpole 

In [ ]:
# Preparation
import numpy as np
import gymnasium as gym
import pygame

 #    Position Velocity_Cart  Winkel Winkelgeschwindigkeit
bins  = [6, 8, 25, 15]    #bins festlegen für die Diskretisierung       
lows  = [-2.4, -3.0, -0.21, -3.5]
highs = [ 2.4,  3.0,  0.21,  3.5]

# kanten für die diskretisierung
def kanten(bereichmin,bereichmax, division):
    kanten=np.linspace (bereichmin,bereichmax, division + 1)
    return kanten

# alle kanten in eine liste packen
alle_kanten = [kanten(lows[i], highs[i], bins[i]) for i in range(4)]

# Diskretisierung denn sonst läuft der bereich der winkel gegen unendlich
def discretize(s):
    idx = []
    for i in range(4):
        b = np.digitize(s[i], alle_kanten[i][1:-1])
        b = min(b, bins[i] - 1)
        idx.append(b)
    return tuple(idx)

q_table = np.zeros(bins + [2] )
alpha= 0.1 #lernrate
gamma=0.99 # disoount

#empfohlene werte
epsilon=1 
epsilon_minimum= 0.02
epsilon_decay= 0.997    

episodes=6000

env = gym.make('CartPole-v1')
state, info = env.reset()

running = True
for episode in range(episodes):
    raw_state, info = env.reset()
    state = discretize(raw_state)
    done=False
    while not done:
        if np.random.random()< epsilon:
                    action= env.action_space.sample()  
        else:
            action=np.argmax(q_table[state]) 

        raw_state, reward, terminated, truncated, info = env.step(action)
        new_state = discretize(raw_state)

        done = terminated or truncated

        td= reward + gamma * np.max(q_table[new_state]) - q_table[state +(action,)] #berechnung des TD errors
        q_table[state + (action,)] += alpha * td

        state= new_state

        if terminated or truncated:
                obs, info = env.reset()

    epsilon = max(epsilon_minimum, epsilon * epsilon_decay)
          
np.save("q_table_cartpole.npy", q_table)
env.close()

done = terminated or truncated


## Testing Cartpole 

In [ ]:
import gymnasium as gym
import numpy as np
import pygame


bins  = [6, 8, 25, 15]
lows  = [-2.4, -3.0, -0.21, -3.5]
highs = [ 2.4,  3.0,  0.21,  3.5]

def kanten(bereichmin, bereichmax, division):
    return np.linspace(bereichmin, bereichmax, division + 1)

alle_kanten = [kanten(lows[i], highs[i], bins[i]) for i in range(4)]

def discretize(s):
    idx = []
    for i in range(4):
        b = np.digitize(s[i], alle_kanten[i][1:-1])
        b = min(b, bins[i] - 1)
        idx.append(b)
    return tuple(idx)

# --- Tabelle laden, Fenster an ---
q_table = np.load('q_table_cartpole.npy')
env = gym.make('CartPole-v1', render_mode='human')

raw_state, info = env.reset()
state = discretize(raw_state)          

running = True
while running:
    action = np.argmax(q_table[state])                       
    raw_state, reward, terminated, truncated, info = env.step(action)
    state = discretize(raw_state)                            

    if terminated or truncated:
        raw_state, info = env.reset()
        state = discretize(raw_state)                        

    for event in pygame.event.get():
        if event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE:
            running = False

env.close()

# Cartpole mit einem Neuronalen Netwerk

In [ ]:
import pygame 
import numpy as np 
import torch.nn as nn  
import gymnasium as gym
import torch
import random
from collections import deque




class QNetwork(nn.Module):
    def __init__(self, n_states=4,n_actions=2, n_h_layers=128 ):
        super().__init__()
        # Netzwerk Definieren
        self.net == nn.Sequential(
            nn.Linear(n_states, n_h_layers ),
            nn.ReLU(), # Nicht-Linearität einfügen und alles im negativen bereich wird auf 0 gesetzt
            nn.Linear(n_h_layers,n_h_layers),
            nn.ReLU(),
            nn.Linear(n_h_layers, n_actions),     
        )   

    def forward(self, x): # bei module(states) wird diese funktion direkt aufgerufen 
        return self.net(x)


# buffer der es ermöglicht es spääter mit random alten erfahrungen zu lernen
class ReplayBuffer:
    def __init__(self, cap=50000):
        self.buffer = deque(maxlen=cap)
    def push(self, *e): # *e fügt alles in ein packet -> buffer.push(state, action, reward, next_state, done)
        self.buffer.append(e)
    def sample(self, b):
        s, a, r, ns, d = zip(*b) # fügt alles einem zip dazu
        return (torch.tensor(np.array(s), dtype=torch.float32),          # alles in torch umwandeln
                torch.tensor(a, dtype=torch.int64),
                torch.tensor(r, dtype=torch.float32),
                torch.tensor(np.array(ns), dtype=torch.float32),
                torch.tensor(d, dtype=torch.float32))

    def __len__(self): return len(self.buffer)   

# HYPERPARAMETER 
EPISODES=600; GAMMA=0.99; LR=1e-3; BATCH_SIZE=64
EPSILON_START=1.0; EPSILON_MIN=0.02; EPSILON_DECAY=0.995; TARGET_UPDATE=10

#--Setup----------------------------------------------------------------
env = gym.make('CartPole-v1')
online_net = QNetwork() ; target_net = QNetwork()    # Hier wird unterschieden zwischen dem laufend updatendes und das zielnetz
target_net.load_state_dict(online_net.state_dict())
optimizer = torch.optim.Adam(online_net.parameters(), lr=LR)


    




        






        



    




In [44]:
from stable_baselines3 import DQN
import gymnasium as gym
 
env = gym.make('CartPole-v1')
model = DQN('MlpPolicy', env, verbose=1)
model.learn(total_timesteps=100000)


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 19.5     |
|    ep_rew_mean      | 19.5     |
|    exploration_rate | 0.993    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 11027    |
|    time_elapsed     | 0        |
|    total_timesteps  | 78       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 18       |
|    ep_rew_mean      | 18       |
|    exploration_rate | 0.986    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 1086     |
|    time_elapsed     | 0        |
|    total_timesteps  | 144      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.542    |
|    n_updates        | 10       |
-------------------------------